# Load epub book

In [1]:
# Import libraries
import os
from langchain_community.document_loaders import UnstructuredEPubLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

import chromadb
from uuid import uuid4
from chromadb.utils import embedding_functions

from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

W0520 08:00:42.340000 34772 torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W0520 08:00:42.365000 34772 torch/utils/_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


In [2]:
# TODO: Load document
chunk_size=1024
chunk_overlap=100

text_splitter = RecursiveCharacterTextSplitter(chunk_size=chunk_size, chunk_overlap=chunk_overlap)

epub_loader = UnstructuredEPubLoader(file_path='/content/docs/charles-dickens_a-christmas-carol.epub')

In [3]:
# TODO Split document
chunks = epub_loader.load_and_split(text_splitter)
print(len(chunks))

[WARNING] Sandbox argument was used, but pandoc version is too low. Ignoring argument.


204


In [4]:
# TODO Examine chunk
idx = 50
print(chunks[idx].page_content)
print(chunks[idx].metadata)


It was a strange figure﻿—like a child; yet not so like a child as like an old man, viewed through some supernatural medium, which gave him the appearance of having receded from the view, and being diminished to a child’s proportions. Its hair, which hung about its neck and down its back, was white, as if with age; and yet the face had not a wrinkle in it, and the tenderest bloom was on the skin. The arms were very long and muscular; the hands the same, as if its hold were of uncommon strength. Its legs and feet, most delicately formed, were, like those upper members, bare. It wore a tunic of the purest white; and round its waist was bound a lustrous belt, the sheen of which was beautiful. It held a branch of fresh green holly in its hand; and, in singular contradiction of that wintry emblem, had its dress trimmed with summer flowers. But the strangest thing about it was, that from the crown of its head there sprang a bright clear jet of light, by which all this was visible; and which w

# Create embeddings

In [5]:
# TODO: Create embedding model
embed_model_name = "BAAI/bge-small-en-v1.5"
#embed_model_name = "all-MiniLM-L6-v2"
embed_function = embedding_functions.SentenceTransformerEmbeddingFunction(model_name=embed_model_name)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [11]:
# TODO: Explore embedding model
text = [ 'hello, world', 'TorchCodec is a Python library for decoding video and audio data into PyTorch tensors, on CPU and CUDA GPU. It also supports video and audio encoding on CPU! It aims to be fast, easy to use, and well integrated into the PyTorch ecosystem. If you want to use PyTorch to train ML models on videos and audio, TorchCodec is how you turn these into data.']

embeddings = embed_function(text)
print(len(embeddings[0]))
print(len(embeddings[1]))
print(embeddings[0])

384
384
[-3.15819047e-02 -4.86476645e-02  3.21324095e-02 -6.57483488e-02
 -1.12420879e-03  1.14271799e-02 -1.62236346e-03  5.49600758e-02
  4.48704287e-02 -2.09958665e-03  7.87422992e-03 -2.20074505e-02
  3.43550555e-02  6.57045990e-02  2.98711974e-02 -2.77359504e-04
  1.02014421e-03 -3.47684957e-02 -1.21079296e-01 -1.47990528e-02
  9.72586572e-02  3.53694819e-02 -1.68968756e-02 -4.28635441e-02
 -2.48042066e-02  5.63814165e-03  6.80473307e-03  1.35494387e-02
  6.07590657e-03 -9.83635113e-02 -6.45543784e-02 -1.15323728e-02
  3.96090746e-02  2.41095573e-02  4.54738475e-02 -2.10404806e-02
  2.52141319e-02 -1.03886537e-02 -7.94329494e-02  3.64227779e-03
  4.60232794e-02 -5.09504452e-02  1.40664671e-02 -3.41337710e-03
  1.36135798e-02 -4.93411534e-02  1.70673206e-02  5.47222644e-02
 -2.78037693e-02  4.88187245e-04 -5.45994379e-02 -8.51238705e-03
 -1.97878145e-02 -2.24604411e-03  2.84831468e-02  9.09865052e-02
  7.97384828e-02  2.93895905e-03  4.68927957e-02  8.69193859e-03
  1.88648328e-02 

In [14]:
# TODO: Prepare the chunks for inserting into Chroma
texts = [ d.page_content for d in chunks ]
text_ids = [ str(uuid4())[:8] for _ in range(len(texts))]
print(len(texts))
print(len(text_ids))
print(text_ids)


204
204
['00c7610c', '407c6733', '082115a3', '201c8208', 'f02d1e3a', '6c823fa0', 'db3bf3df', '0ea2473d', 'ef9ba339', 'c82ed0e4', 'b7c42ed4', '0fe03262', '9370e46f', '5353a269', 'e9095571', '8095b187', '606bef76', '6f672665', 'cd2f4c79', '9269ca28', '3f0ca71f', '275c5072', '8b0ef539', 'ddcd5d17', '4ba996a4', '88366104', 'c2e4a582', 'e5cf0010', 'a31b176d', '65377853', '487a1502', '7101c6eb', '0625c45d', '6827cfe0', '5e1f369e', '4dd6f67b', '10827f1f', '08d05b0d', 'f5953c47', '0835490c', '7bc37848', 'adbb84a8', '39c8104c', '93a18637', 'b90176ca', '0fe77412', '6afdc641', 'da68bb70', '5811e8f3', '23d37249', '4aa4303d', '682d7cf2', 'b7f1e230', 'b44734d1', '3389cfe5', 'd5e6c265', 'f021637e', '4e286fab', 'd3863c05', 'c5a21623', '3a5f11c5', '0c175a59', '5242beea', '46e94545', '7677c761', '3416b56d', '662db331', '28acd813', 'b9d0ea55', '6c027c82', '18c27ab0', '4f410ad6', 'a1132116', 'f973296d', '25cb73a9', 'e80442a5', 'ea42346d', 'c47f170b', 'edd2ad31', '700c5056', 'e433685f', '528834b3', '7c1dd0

In [15]:
# TODO: Create ephemeral Chroma client and save chunks
col_name = 'a-christmas-carol'

ch_client = chromadb.Client()

# clean the database
try:
  ch_client.delete_collection(col_name)
except:
  pass

# create the collection
ch_collection = ch_client.create_collection(
    name=col_name,
    embedding_function=embed_function
)


In [16]:
# Insert the documents into the database
ch_collection.add(
    documents=texts,
    ids=text_ids
)

In [17]:
# TODO: Print number of documents in collection
print(ch_collection.count())

204


In [21]:
# TODO: Query collection
query = "The ghost of christmas past is the first ghost to appear before Scrooge"

results = ch_collection.query(
    query_texts=query,
    n_results=5
)

for k, v in results.items():
  print(f'{k}: {v}')

ids: [['d89c4008', '0075104c', '1f47c6db', 'b7f1e230', '8029dd26']]
embeddings: None
documents: [['“Come in!” exclaimed the Ghost. “Come in! and know me better, man!”\n\nScrooge entered timidly, and hung his head before this Spirit. He was not the dogged Scrooge he had been; and though the Spirit’s eyes were clear and kind, he did not like to meet them.\n\n“I am the Ghost of Christmas Present,” said the Spirit. “Look upon me!”', '“I am the Ghost of Christmas Present,” said the Spirit. “Look upon me!”\n\nScrooge reverently did so. It was clothed in one simple deep green robe, or mantle, bordered with white fur. This garment hung so loosely on the figure, that its capacious breast was bare, as if disdaining to be warded or concealed by any artifice. Its feet, observable beneath the ample folds of the garment, were also bare; and on its head it wore no other covering than a holly wreath, set here and there with shining icicles. Its dark-brown curls were long and free; free as its genial f

In [23]:
for id in results['ids'][0]:
  r = ch_collection.get(id)
  print(r['documents'][0])
  print('===================================')

“Come in!” exclaimed the Ghost. “Come in! and know me better, man!”

Scrooge entered timidly, and hung his head before this Spirit. He was not the dogged Scrooge he had been; and though the Spirit’s eyes were clear and kind, he did not like to meet them.

“I am the Ghost of Christmas Present,” said the Spirit. “Look upon me!”
“I am the Ghost of Christmas Present,” said the Spirit. “Look upon me!”

Scrooge reverently did so. It was clothed in one simple deep green robe, or mantle, bordered with white fur. This garment hung so loosely on the figure, that its capacious breast was bare, as if disdaining to be warded or concealed by any artifice. Its feet, observable beneath the ample folds of the garment, were also bare; and on its head it wore no other covering than a holly wreath, set here and there with shining icicles. Its dark-brown curls were long and free; free as its genial face, its sparkling eye, its open hand, its cheery voice, its unconstrained demeanour, and its joyful air. Gi

# Question and Answer LLM
In this exercise you will implement a question and answer LLM for the 'A Christmas Carol' book that you have chunked and saved.

The workflow is as follows:
1. Assume you ask the following question regarding the book eg. `"Who is Scrooge?"`?
2. Query the relevant context from Chroma with the question or facts from the question.
3. Combine the question and the top 5 context return by Chroma into a prompt
4. Use `google/flan-t5-base` to answer the question.

Look through the FLAN templates in [Github](https://github.com/google-research/FLAN/blob/main/flan/templates.py) and select an appropriate template for this workshop.

Do not worry about the accuracy of the result. Focus on implementing the solution. We will discuss the nuances of the solution at the end of the workshop.

Use your RAG workflow to answer the provided questions in `questions_for_rag.txt` file.

In [ ]:
# TODO Your code

In [ ]:
# TODO Your code

In [ ]:
# TODO Your code

# Discussion

1. How did your solution perform?
2. Where do you think are the issues?
3. How can you improve it?